# 10 — 实例生命周期管理

`IviumsoftInstanceManager` 启动、跟踪、接管并关闭 IviumSoft 进程，将每个操作系统进程
映射到它注册时获得的**驱动实例号**。它是笔记本 `02` 中实例*作用域*的对应部分：作用域决定
命令作用于*哪个*正在运行的实例；管理器决定*存在哪些实例*。

### 何时需要它

- 以编程方式启动 N 个 IviumSoft 窗口并并行驱动它们
- 崩溃后恢复：重新接管（`adopt`）在脚本之外存活下来的实例
- 清理上一次运行遗留的游离 IviumSoft 进程

### 注意

- **仅支持 Windows** — 使用原生 Win32 进程辅助函数。
- 驱动必须已打开。在**尚无 IviumSoft 运行**的冷启动场景下，用
  `Pyvium.open_driver(verify_iviumsoft=False)` 打开，以便管理器启动第一个实例。
- 关闭后驱动的实例编号保持稳定：关闭其中一个会留下空缺，而不会重新编号其余实例。

> 本笔记本驱动真实进程，因此其单元格未预先执行。请在安装了 IviumSoft 的 Windows 机器上运行。

In [ ]:
from pyvium import Pyvium, IviumsoftInstanceManager
print("实例管理器已导入")

## 1. 冷启动

在无需运行中的 IviumSoft 的情况下打开驱动，然后创建管理器。如果你的 IviumSoft 不在默认路径
`C:\\IviumStat\\IviumSoft.exe`，请传入 `exe_path=...`。

In [ ]:
Pyvium.open_driver(verify_iviumsoft=False)

manager = IviumsoftInstanceManager(
    # exe_path=r"C:\IviumStat\IviumSoft.exe",   # 若安装在其他路径请覆盖
    launch_timeout=30.0,
    close_timeout=10.0,
)
print("驱动已打开（冷启动）；管理器就绪")
print("已活跃的实例:", Pyvium.get_active_iviumsoft_instances())

## 2. 启动实例

`launch()` 启动一个 IviumSoft 进程并阻塞，直到它向驱动注册，然后返回其 `ManagedInstance`
（实例号 + pid + 启动时间）。它在内部串行化启动，因此即使你连续启动多个，实例号也会被正确归属。

In [ ]:
first = manager.launch()
print("已启动:", first)

second = manager.launch()
print("已启动:", second)

print("实例号:", first.instance_number, second.instance_number)

## 3. 列出实例

`list_instances()` 为每个活跃的驱动实例返回一条记录。受管理的/已接管的带有 pid；本管理器未
打开的实例（孤儿）返回 `pid=None`。过期记录（进程已消失）会被清除。

In [ ]:
for record in manager.list_instances():
    kind = "受管理" if record.managed else ("已接管/孤儿")
    print(f"  实例 {record.instance_number}: pid={record.pid} ({kind})")

## 4. 驱动一个受管理的实例

管理器只负责生命周期；发送命令请使用笔记本 `02` 的作用域 API。这里我们在第一个已启动的实例上
连接设备并读取其状态。

In [ ]:
handle = Pyvium.instance(first.instance_number)

status, label = handle.get_device_status()
print(f"实例 {first.instance_number}: 状态 ({status}, '{label}')")

if status == 0:  # IviumSoft 已启动，但设备尚未连接
    try:
        handle.connect_device()
        print("  已连接, 序列号:", handle.get_device_serial_number())
    except Exception as error:
        print(f"  跳过连接: {type(error).__name__}: {error}")

## 5. Discover：核对驱动视图与操作系统视图

`discover()` 是只读的。它将管理器所跟踪的与实际运行的进行配对，并分离出无法自动配对的两半：
`orphan_instance_numbers`（驱动侧）和 `untracked_processes`（操作系统侧）。健康状态下两者数量一致。

In [ ]:
report = manager.discover()
print("已跟踪:")
for record in report.tracked:
    print(f"  实例 {record.instance_number} (pid {record.pid})")
print("孤儿实例号:", report.orphan_instance_numbers)
print("未跟踪进程:")
for process in report.untracked_processes:
    print(f"  pid {process.pid}  启动时间 {process.started_at}  标题 {process.window_title!r}")

## 6. 接管一个并非本管理器启动的实例

脚本重启后，IviumSoft 窗口仍在运行，但本管理器已无它们的记录。`adopt(instance_number, pid)`
可重新关联。pid 必须来自记录了它的外部来源，因为驱动无法将实例号映射到 pid；`discover()` 可
帮助你按启动顺序配对（驱动按顺序为实例编号）。

In [ ]:
# 示例恢复流程：将每个未跟踪进程（最早的在前）与最小的孤儿实例号配对，然后接管它。
# 取消注释以对真实孤儿运行。
#
# report = manager.discover()
# for instance_number, process in zip(report.orphan_instance_numbers,
#                                     report.untracked_processes):
#     record = manager.adopt(instance_number, process.pid)
#     print("已接管:", record)
print("adopt() 通过 (instance_number, pid) 重新关联一个已存在的实例")

## 7. 关闭一个实例

`close()` 发送一次优雅的窗口关闭，若进程未在 `close_timeout` 内退出则升级为强制终止。除非
`force=True`，否则拒绝关闭*正在测量*的实例。只有具有已知 pid（已启动或已接管）的实例才能在此关闭。

In [ ]:
manager.close(second.instance_number)
print("已关闭实例", second.instance_number)
print("剩余:", [r.instance_number for r in manager.list_instances()])

## 8. 清扫孤儿进程

`close_orphans()` 优雅地关闭所有本管理器未跟踪的 IviumSoft 进程（即 `discover()` 的
`untracked_processes`）。若任一孤儿实例正在测量，则除非 `force=True` 否则不关闭任何进程 —
对于仍在使用的实例，请优先使用 `discover()` + `adopt()`。

In [ ]:
# closed_pids = manager.close_orphans()          # 加 force=True 可强制关闭正忙的进程
# print("已关闭的孤儿 pid:", closed_pids)
print("close_orphans() 会清扫未跟踪的 IviumSoft 进程")

## 清理

关闭本笔记本启动的一切，然后关闭驱动。

In [ ]:
for record in manager.list_instances():
    if record.managed:
        try:
            manager.close(record.instance_number, force=True)
            print("已关闭", record.instance_number)
        except Exception as error:
            print(f"关闭 {record.instance_number} 失败: {type(error).__name__}: {error}")

Pyvium.close_driver()
print("驱动已关闭")

---

## 小结

| 任务 | API |
|------|-----|
| 冷启动打开（尚无 IviumSoft） | `Pyvium.open_driver(verify_iviumsoft=False)` |
| 创建管理器 | `IviumsoftInstanceManager(exe_path=..., ...)` |
| 启动一个实例 | `.launch()` -> `ManagedInstance` |
| 列出活跃实例 | `.list_instances()` |
| 核对驱动 vs 操作系统 | `.discover()` -> `DiscoveryReport` |
| 重启后重新关联 | `.adopt(instance_number, pid)` |
| 关闭一个实例 | `.close(instance_number, force=False)` |
| 清扫未跟踪进程 | `.close_orphans(force=False)` |

## 下一步

- **`02_device_and_instance_management.ipynb`** — 将命令限定到某个实例 / 通道
- **`08_batch_and_synchronization.ipynb`** — 跨实例协调测量